# 05. Memory Systems para Agentes

**Nivel:** 🔴 Avanzado  
**Tiempo estimado:** 90 minutos  
**Prerequisitos:** [01-04: Intro, Prompting, ReAct, Tool Use](01-intro-llm-agents.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Implementar diferentes tipos de memoria (short-term, long-term)
- Usar vectores embeddings para retrieval semántico
- Construir sistemas RAG (Retrieval-Augmented Generation) para agentes
- Integrar bases de datos vectoriales (FAISS, Chroma)
- Manejar contextos largos con windowing y summarization
- Diseñar estrategias de memoria para conversaciones multi-turno

## 1. Motivación: Agentes Que Recuerdan

### El Problema: Context Window Limitado

**Escenario:**
```
Usuario: "Mi nombre es Alice y me gusta el jazz"
Agente: "Hola Alice, ¿en qué puedo ayudarte?"

[30 mensajes después...]

Usuario: "¿Recuerdas mi nombre?"
Agente: "Lo siento, no tengo esa información" ❌
```

**Problemas:**
- Context window limitado (4K-128K tokens)
- Información importante se "olvida" en conversaciones largas
- No hay persistencia entre sesiones
- Costos escalan con contexto

### La Solución: Memory Systems

**Tipos de Memoria:**

1. **Short-Term (Buffer)**: Últimos N mensajes
2. **Summarization**: Resúmenes progresivos
3. **Entity Memory**: Hechos sobre entidades
4. **Vector Memory (RAG)**: Retrieval semántico
5. **Long-Term (DB)**: Persistencia permanente

### Pregunta Guía

**Al final responderemos:**
*¿Cómo diseñar sistemas de memoria que permitan a agentes mantener contexto coherente en conversaciones largas y entre sesiones?*

In [ ]:
# Instalación
# !pip install openai faiss-cpu chromadb sentence-transformers numpy

import os
import json
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
from datetime import datetime
from collections import deque
import numpy as np

# Embeddings
try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMERS_AVAILABLE = True
except ImportError:
    SENTENCE_TRANSFORMERS_AVAILABLE = False
    print("⚠️  sentence-transformers no disponible")

# Vector DB
try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False
    print("⚠️  FAISS no disponible")

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False

from dotenv import load_dotenv
load_dotenv()

print("✅ Librerías importadas")

## 2. Arquitectura de Memory Systems

```
┌────────────────────────────────────────────────────────┐
│             MEMORY ARCHITECTURE                        │
├────────────────────────────────────────────────────────┤
│                                                        │
│  ┌─────────────────────────────────────────────────┐  │
│  │        Short-Term Memory (Buffer)               │  │
│  │  - Last N messages                              │  │
│  │  - Fast access                                  │  │
│  │  - Conversation context                         │  │
│  └─────────────────────────────────────────────────┘  │
│                      ↓                                 │
│  ┌─────────────────────────────────────────────────┐  │
│  │        Summarization Memory                     │  │
│  │  - Progressive summaries                        │  │
│  │  - Condensed context                            │  │
│  └─────────────────────────────────────────────────┘  │
│                      ↓                                 │
│  ┌─────────────────────────────────────────────────┐  │
│  │        Vector Memory (RAG)                      │  │
│  │  - Embeddings + Vector DB                       │  │
│  │  - Semantic search                              │  │
│  │  - Retrieval by relevance                       │  │
│  └─────────────────────────────────────────────────┘  │
│                      ↓                                 │
│  ┌─────────────────────────────────────────────────┐  │
│  │        Long-Term Memory (Database)              │  │
│  │  - Persistent storage                           │  │
│  │  - Cross-session                                │  │
│  │  - Structured data                              │  │
│  └─────────────────────────────────────────────────┘  │
│                                                        │
└────────────────────────────────────────────────────────┘
```

## 3. Implementación: Memory Types

In [ ]:
@dataclass
class Message:
    """Representa un mensaje en la conversación"""
    role: str  # "user" o "assistant"
    content: str
    timestamp: datetime = field(default_factory=datetime.now)
    metadata: Dict = field(default_factory=dict)

class BufferMemory:
    """
    Short-term memory: mantiene últimos N mensajes.
    """
    
    def __init__(self, max_messages: int = 10):
        self.max_messages = max_messages
        self.messages: deque = deque(maxlen=max_messages)
    
    def add(self, role: str, content: str, metadata: Dict = None):
        """Agrega mensaje al buffer"""
        msg = Message(role, content, metadata=metadata or {})
        self.messages.append(msg)
    
    def get_messages(self) -> List[Dict]:
        """Retorna mensajes en formato para LLM"""
        return [
            {"role": msg.role, "content": msg.content}
            for msg in self.messages
        ]
    
    def clear(self):
        """Limpia el buffer"""
        self.messages.clear()

print("✅ BufferMemory implementada")

In [ ]:
class SummarizationMemory:
    """
    Mantiene resumen de la conversación para reducir tokens.
    """
    
    def __init__(self, llm_client=None, summarize_every: int = 10):
        self.llm_client = llm_client
        self.summarize_every = summarize_every
        self.messages = []
        self.summary = ""
        self.message_count = 0
    
    def add(self, role: str, content: str):
        """Agrega mensaje y resume si es necesario"""
        self.messages.append({"role": role, "content": content})
        self.message_count += 1
        
        # Resumir cada N mensajes
        if self.message_count % self.summarize_every == 0:
            self._create_summary()
    
    def _create_summary(self):
        """Crea resumen de conversación reciente"""
        if not self.messages:
            return
        
        # Crear prompt para resumen
        conversation = "\n".join([
            f"{msg['role']}: {msg['content']}"
            for msg in self.messages
        ])
        
        summary_prompt = f"""Resume la siguiente conversación de forma concisa, 
capturando los puntos clave y hechos importantes:

{conversation}

Resumen:"""
        
        # Generar resumen (simplificado para demo)
        new_summary = f"[Resumen de {len(self.messages)} mensajes: {self.messages[-1]['content'][:50]}...]"
        
        # Combinar con resumen anterior
        if self.summary:
            self.summary = f"{self.summary}\n{new_summary}"
        else:
            self.summary = new_summary
        
        # Limpiar mensajes resumidos
        self.messages = []
    
    def get_context(self) -> str:
        """Retorna contexto (resumen + mensajes recientes)"""
        context_parts = []
        
        if self.summary:
            context_parts.append(f"Resumen de conversación previa:\n{self.summary}")
        
        if self.messages:
            recent = "\n".join([
                f"{msg['role']}: {msg['content']}"
                for msg in self.messages
            ])
            context_parts.append(f"Mensajes recientes:\n{recent}")
        
        return "\n\n".join(context_parts)

print("✅ SummarizationMemory implementada")

In [ ]:
class VectorMemory:
    """
    Vector-based memory con semantic search (RAG).
    """
    
    def __init__(self, embedding_model: str = "all-MiniLM-L6-v2"):
        self.memories = []  # Lista de (text, embedding, metadata)
        
        # Cargar modelo de embeddings
        if SENTENCE_TRANSFORMERS_AVAILABLE:
            self.encoder = SentenceTransformer(embedding_model)
            self.dimension = 384  # Dimensión de all-MiniLM-L6-v2
        else:
            self.encoder = None
            self.dimension = 384
        
        # Inicializar FAISS index
        if FAISS_AVAILABLE:
            self.index = faiss.IndexFlatL2(self.dimension)
        else:
            self.index = None
    
    def add(self, text: str, metadata: Dict = None):
        """Agrega memoria con embedding"""
        if self.encoder:
            embedding = self.encoder.encode([text])[0]
        else:
            # Embedding simulado
            embedding = np.random.rand(self.dimension).astype('float32')
        
        self.memories.append({
            "text": text,
            "embedding": embedding,
            "metadata": metadata or {},
            "timestamp": datetime.now()
        })
        
        # Agregar a FAISS index
        if self.index is not None:
            self.index.add(np.array([embedding]))
    
    def search(self, query: str, top_k: int = 3) -> List[Dict]:
        """Busca memorias más relevantes por similitud semántica"""
        if not self.memories:
            return []
        
        # Encode query
        if self.encoder:
            query_embedding = self.encoder.encode([query])[0]
        else:
            query_embedding = np.random.rand(self.dimension).astype('float32')
        
        # Buscar en FAISS
        if self.index is not None:
            distances, indices = self.index.search(
                np.array([query_embedding]), 
                min(top_k, len(self.memories))
            )
            
            results = []
            for idx, dist in zip(indices[0], distances[0]):
                if idx < len(self.memories):
                    results.append({
                        "text": self.memories[idx]["text"],
                        "metadata": self.memories[idx]["metadata"],
                        "distance": float(dist)
                    })
            
            return results
        
        # Fallback: retornar primeros k
        return [{
            "text": mem["text"],
            "metadata": mem["metadata"],
            "distance": 0.0
        } for mem in self.memories[:top_k]]

print("✅ VectorMemory implementada")

## 4. Agente con Memory System Completo

In [ ]:
class MemoryAgent:
    """
    Agente con sistema de memoria multi-nivel.
    """
    
    def __init__(self, use_vector_memory: bool = True):
        self.buffer = BufferMemory(max_messages=5)
        self.summarization = SummarizationMemory(summarize_every=10)
        self.use_vector_memory = use_vector_memory
        
        if use_vector_memory:
            self.vector_memory = VectorMemory()
        else:
            self.vector_memory = None
    
    def _build_context(self, query: str) -> str:
        """Construye contexto usando todas las memorias"""
        context_parts = []
        
        # 1. Búsqueda en vector memory
        if self.vector_memory:
            relevant_memories = self.vector_memory.search(query, top_k=3)
            if relevant_memories:
                memories_text = "\n".join([
                    f"- {mem['text']}"
                    for mem in relevant_memories
                ])
                context_parts.append(f"Información relevante recordada:\n{memories_text}")
        
        # 2. Resumen de conversación
        summary_context = self.summarization.get_context()
        if summary_context:
            context_parts.append(summary_context)
        
        # 3. Buffer de mensajes recientes
        recent_messages = self.buffer.get_messages()
        if recent_messages:
            recent_text = "\n".join([
                f"{msg['role']}: {msg['content']}"
                for msg in recent_messages
            ])
            context_parts.append(f"Conversación inmediata:\n{recent_text}")
        
        return "\n\n".join(context_parts)
    
    def chat(self, user_input: str, verbose: bool = True) -> str:
        """Procesa mensaje del usuario con memoria"""
        # Agregar a buffer
        self.buffer.add("user", user_input)
        
        # Agregar a summarization
        self.summarization.add("user", user_input)
        
        # Extraer facts importantes para vector memory
        if self.vector_memory and self._is_important_info(user_input):
            self.vector_memory.add(user_input, {"type": "user_info"})
        
        # Construir contexto
        context = self._build_context(user_input)
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"📝 Contexto construido:")
            print(f"{context}\n")
        
        # Generar respuesta (simulada)
        response = self._generate_response(user_input, context)
        
        # Agregar respuesta a memorias
        self.buffer.add("assistant", response)
        self.summarization.add("assistant", response)
        
        return response
    
    def _is_important_info(self, text: str) -> bool:
        """Detecta si el texto contiene información importante para recordar"""
        keywords = ["mi nombre", "me llamo", "soy", "me gusta", "prefiero", "trabajo"]
        return any(kw in text.lower() for kw in keywords)
    
    def _generate_response(self, user_input: str, context: str) -> str:
        """Genera respuesta usando contexto (simulado)"""
        # En producción, usar LLM real con el contexto
        if "nombre" in user_input.lower() and "recuerdas" in user_input.lower():
            # Buscar nombre en vector memory
            if self.vector_memory:
                memories = self.vector_memory.search("nombre", top_k=1)
                if memories:
                    return f"Sí, recuerdo: {memories[0]['text']}"
        
        return f"He procesado tu mensaje con el contexto completo. Tengo {len(self.buffer.messages)} mensajes recientes en memoria."

print("✅ MemoryAgent implementado")

## 5. Demo: Agente con Memoria

In [ ]:
# Crear agente con memoria
agent = MemoryAgent(use_vector_memory=True)

# Conversación de prueba
print("\n🤖 Conversación con Memory Agent\n")

responses = [
    agent.chat("Hola, mi nombre es Alice y me gusta el jazz", verbose=False),
    agent.chat("¿Qué tiempo hace hoy?", verbose=False),
    agent.chat("Trabajo como ingeniera de software", verbose=False),
]

for i, resp in enumerate(responses, 1):
    print(f"Mensaje {i}: {resp}\n")

# Probar memoria
print("\n--- Probando memoria ---\n")
memory_test = agent.chat("¿Recuerdas mi nombre y qué me gusta?", verbose=True)
print(f"\nRespuesta: {memory_test}")

## 6. RAG (Retrieval-Augmented Generation)

RAG combina retrieval de documentos con generación de LLM.

In [ ]:
class RAGAgent:
    """
    Agente con RAG: recupera documentos relevantes antes de generar respuesta.
    """
    
    def __init__(self, documents: List[str]):
        self.vector_store = VectorMemory()
        
        # Indexar documentos
        for i, doc in enumerate(documents):
            self.vector_store.add(doc, metadata={"doc_id": i})
        
        print(f"✅ Indexados {len(documents)} documentos")
    
    def query(self, question: str, top_k: int = 3, verbose: bool = True) -> str:
        """Responde pregunta usando RAG"""
        # 1. Retrieve: Buscar documentos relevantes
        relevant_docs = self.vector_store.search(question, top_k=top_k)
        
        if verbose:
            print(f"\n📚 Documentos recuperados:")
            for i, doc in enumerate(relevant_docs, 1):
                print(f"  {i}. {doc['text'][:100]}...")
        
        # 2. Augment: Construir prompt con contexto
        context = "\n\n".join([doc["text"] for doc in relevant_docs])
        
        prompt = f"""Responde la pregunta basándote SOLO en el siguiente contexto:

Contexto:
{context}

Pregunta: {question}

Respuesta:"""
        
        # 3. Generate: Generar respuesta (simulado)
        response = f"Basándome en los documentos, puedo decir que... [Aquí iría respuesta generada por LLM usando el contexto]"
        
        return response

# Documentos de ejemplo
knowledge_base = [
    "Python es un lenguaje de programación de alto nivel creado por Guido van Rossum en 1991.",
    "Los agentes de IA pueden usar herramientas externas mediante function calling.",
    "RAG (Retrieval-Augmented Generation) combina búsqueda de documentos con generación de texto.",
    "Las bases de datos vectoriales como FAISS permiten búsqueda semántica eficiente.",
    "La memoria a corto plazo de los agentes almacena conversaciones recientes."
]

rag_agent = RAGAgent(knowledge_base)
answer = rag_agent.query("¿Qué es RAG?", verbose=True)
print(f"\n✅ Respuesta: {answer}")

## 7. Ejercicios

### 🟢 Ejercicio 1: Entity Memory

In [ ]:
def ejercicio_1_entity_memory():
    """
    Objetivo: Implementar memoria de entidades (personas, lugares, hechos)
    
    Instrucciones:
    1. Crea clase EntityMemory que extraiga entidades de mensajes
    2. Almacena hechos sobre cada entidad
    3. Recupera info de entidades cuando se mencionen
    
    Ejemplo:
    - "Alice vive en Madrid" → Entity: Alice, Fact: lives_in=Madrid
    - "¿Dónde vive Alice?" → Retrieval: Alice.lives_in
    """
    # TODO: Tu código aquí
    pass

### 🟡 Ejercicio 2: Hybrid Search

Combina keyword search + semantic search.

In [ ]:
def ejercicio_2_hybrid_search():
    """
    Objetivo: Implementar búsqueda híbrida
    
    Idea:
    - Semantic search (embeddings) captura similitud semántica
    - Keyword search (BM25) captura matches exactos
    - Combinar scores con weighted average
    
    Instrucciones:
    1. Implementa BM25 básico para keyword search
    2. Combina con vector search
    3. Experimenta con diferentes pesos (α semantic, β keyword)
    """
    # TODO: Tu código aquí
    pass

### 🔴 Ejercicio 3: Conversational RAG

RAG que mantiene contexto de conversación.

In [ ]:
def ejercicio_3_conversational_rag():
    """
    Objetivo: RAG que entiende follow-up questions
    
    Ejemplo:
    User: "¿Qué es Python?"
    Agent: [responde basado en docs]
    
    User: "¿Quién lo creó?"  ← Necesita entender "lo" = Python
    Agent: [responde sobre creador]
    
    Instrucciones:
    1. Combina RAGAgent con BufferMemory
    2. Reformula queries usando contexto conversacional
    3. Mantén coherencia en multi-turn conversation
    """
    # TODO: Tu código aquí
    pass

## 8. Resumen y Recursos

### 📚 Resumen

- **Buffer Memory**: Short-term, últimos N mensajes
- **Summarization**: Resúmenes progresivos para conversaciones largas
- **Vector Memory**: Retrieval semántico con embeddings
- **RAG**: Combina retrieval + generation
- **Hybrid Approach**: Múltiples tipos de memoria trabajan juntos

### 🔗 Recursos

#### 📄 Papers

1. **"Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"** (Lewis et al., 2020)
2. **"MemGPT: Towards LLMs as Operating Systems"** (Packer et al., 2023)

#### 💻 Herramientas

- **LangChain Memory**: [Docs](https://python.langchain.com/docs/modules/memory/)
- **ChromaDB**: Vector database
- **Pinecone, Weaviate**: Managed vector DBs

### ➡️ Próximo Paso

**[➡️ Ir al Notebook 06: Sistemas Multi-Agente](06-multi-agentes.ipynb)**

</div>